# 05 - Ablation Study
**AC-PINN Project** | Authors: Suyash Vasal Jain, Nishita Raghvendra, Priyal Agrawal

Isolates the contribution of each AC-PINN component on Burgers' equation
under noisy sparse data (the hardest condition).

| Model | Curriculum | Adaptive Weights |
|---|---|---|
| Vanilla PINN | ✗ | ✗ |
| + Curriculum only | ✓ | ✗ |
| + Adaptive Weights (ratio) only | ✗ | ratio |
| + Adaptive Weights (gradient) only | ✗ | gradient |
| Full AC-PINN (ratio) | ✓ | ratio |
| Full AC-PINN (gradient) | ✓ | gradient |
| **Full AC-PINN (both)** | ✓ | both |

In [ ]:
import sys, os
sys.path.append('..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from pinn_base import (
    device, NoisyDataGenerator, PINNSolver, ACPINNSolver,
    BurgersFDM, Benchmark, save_metrics, save_history
)

PDE        = 'burgers'
LAYERS     = [2, 64, 64, 64, 64, 64, 1]
EPOCHS     = 10000
PDE_PARAMS = {'nu': 0.01/np.pi}
RESULTS    = '../results/burgers/'
FIGURES    = '../figures/burgers/'
os.makedirs(RESULTS, exist_ok=True)

gen  = NoisyDataGenerator(pde=PDE, **PDE_PARAMS)
# Hardest condition - noisy + sparse
data = gen.generate(N_ic=20, N_bc=20, N_f=2000, noise_eps=0.1)

fdm  = BurgersFDM(nx=256, nt=2000, nu=0.01/np.pi)
fdm.solve()
print(f'Device: {device} | Data ready')

## Model 1 - Vanilla PINN (no curriculum, no adaptive weights)

In [ ]:
m1 = PINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS)
h1 = m1.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h1, RESULTS+'ablation_vanilla.npy')

## Model 2 - Curriculum Only (no adaptive weights)

In [ ]:
# ACPINNSolver with ratio strategy but effectively fixed weights
# We set weight_update_every very high to disable adaptive updates
m2 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='ratio')
h2 = m2.fit(data, epochs=EPOCHS, print_every=2000, weight_update_every=999999)
save_history(h2, RESULTS+'ablation_curriculum_only.npy')

## Model 3 - Adaptive Weights (ratio) Only, no curriculum

In [ ]:
# Disable curriculum by using very small resample pool
# and always sampling uniformly (stage 4 from epoch 0)
m3 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='ratio', N_pool=2000, resample_every=1)
# Force stage 4 always by patching sampler thresholds
m3.sampler.STAGE_THRESHOLDS = [0.0, 0.0, 0.0, 1.0]
h3 = m3.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h3, RESULTS+'ablation_ratio_weights_only.npy')

## Model 4 - Adaptive Weights (gradient) Only, no curriculum

In [ ]:
m4 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='gradient', N_pool=2000, resample_every=1)
m4.sampler.STAGE_THRESHOLDS = [0.0, 0.0, 0.0, 1.0]
h4 = m4.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h4, RESULTS+'ablation_gradient_weights_only.npy')

## Model 5 - Full AC-PINN (ratio)

In [ ]:
m5 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='ratio', N_pool=20000, resample_every=500)
h5 = m5.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h5, RESULTS+'ablation_full_ratio.npy')

## Model 6 - Full AC-PINN (gradient)

In [ ]:
m6 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='gradient', N_pool=20000, resample_every=500)
h6 = m6.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h6, RESULTS+'ablation_full_gradient.npy')

## Model 7 - Full AC-PINN (both) - Best Configuration

In [ ]:
m7 = ACPINNSolver(pde=PDE, layers=LAYERS, pde_params=PDE_PARAMS,
                   weight_strategy='both', N_pool=20000, resample_every=500)
h7 = m7.fit(data, epochs=EPOCHS, print_every=2000)
save_history(h7, RESULTS+'ablation_full_both.npy')

## Ablation Benchmark Table

In [ ]:
models = [
    ('Vanilla PINN',                    m1),
    ('+ Curriculum only',               m2),
    ('+ Ratio weights only',            m3),
    ('+ Gradient weights only',         m4),
    ('Full AC-PINN (ratio)',            m5),
    ('Full AC-PINN (gradient)',         m6),
    ('Full AC-PINN (both) ← Best',     m7),
]

bench = Benchmark(fdm, nx=200, nt=100)
for name, model in models:
    bench.add(name, model)
bench.run()

ablation_metrics = bench.compare_metrics()
save_metrics(ablation_metrics, RESULTS+'ablation_metrics.npy')

## Ablation Loss Curves

In [ ]:
histories = [h1, h2, h3, h4, h5, h6, h7]
labels    = ['Vanilla', 'Curriculum', 'Ratio W', 'Grad W',
             'AC(ratio)', 'AC(grad)', 'AC(both)']

plt.figure(figsize=(12, 5))
for h, lbl in zip(histories, labels):
    plt.plot(h['total'], label=lbl, alpha=0.8)
plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Total Loss')
plt.title('Ablation Study - Total Loss Curves (Burgers, Noisy Sparse)')
plt.legend(fontsize=8); plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES+'ablation_loss_curves.png', dpi=150)
plt.show()
print('Ablation study complete.')